In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 8.3 Hartree–Fock I: Atoms

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume VIII — Electronic Structure and Many-Body Matter",
    number="8.3",
    title="Hartree–Fock I: Atoms",
    blurb="The first systematic attack on the many-electron problem, derived and then "
    "built: the energy of a Slater determinant assembled term by term, the "
    "Hartree–Fock equations from the variational principle, and a working radial "
    "code that solves helium and beryllium — exchange as a genuinely nonlocal "
    "operator, shells emerging from self-consistency, Koopmans' theorem meeting "
    "experimental ionization energies, and the correlation ledger of real atoms "
    "opened.",
    difficulty="advanced",
    estimate="130–160 min",
)

## Notebook overview

The laboratory of [§8.2](exact-laboratory.ipynb) met Hartree–Fock as a three-line
special case; this notebook derives the method properly and takes it to real atoms.
The idea is the oldest disciplined answer to the wall of
[§8.1](many-electron-problem.ipynb): restrict the wavefunction to a single Slater
determinant — the most general state in which electrons are *uncorrelated* apart
from the antisymmetry quantum mechanics forces on them — and let the variational
principle pick the best orbitals. What comes out is remarkable machinery. The
electron–electron repulsion resolves into two terms with entirely different
characters: a classical Hartree term, the electrostatics of the charge cloud, and an
**exchange** term with no classical reading at all, a *nonlocal* operator born
purely from antisymmetry. Making that nonlocality concrete (an actual dense matrix
coupling every grid point to every other, built and diagonalized) is this notebook's
central computational move.

The program: assemble the determinant's energy and certify the radial integrals
against closed forms; derive the Hartree–Fock equations and solve helium
self-consistently; watch screening emerge in the converged orbital; solve beryllium
with the full nonlocal exchange and watch the $1s/2s$ shell structure appear; test
Koopmans' theorem against measured ionization energies; and close the ledger that
[§8.1](many-electron-problem.ipynb) opened, with correlation energies of real atoms
now computed against our own Hartree–Fock numbers. The successor notebooks take the
two roads out: the electron gas, where exchange can be done exactly and its
pathology exposed ([§8.4](hartree-fock-gas.ipynb)), and density-functional theory,
which trades the nonlocal operator for a local potential and an honest error bar
([§8.7](kohn-sham.ipynb)).

> **Conventions (this notebook).** Hartree atomic units. All atoms here are
> closed-shell with $s$ orbitals only (He: $1s^2$; Be: $1s^2 2s^2$), so every
> orbital is $\phi(\mathbf r) = u(r)/(r\sqrt{4\pi})$ with $u$ on the radial grid
> $r_j = jh$, $h = 25/1400$ Bohr, and the electron–electron kernel reduces to its
> monopole $1/r_>$ (the shell theorem of
> [§3.3](../03-electrodynamics/gauss-law.ipynb), quantum edition). Radial integrals
> use `numpy.trapezoid`; the Fock matrix is dense and diagonalized by
> `scipy.linalg.eigh`; orbital normalization is $\int u^2\,dr = 1$.
>
> **How to read the checks.** Each exercise closes with a `validate` call against an
> independent fact: a closed-form integral, the literature Hartree–Fock limits, the
> virial theorem, a measured ionization energy. A ✓ is strong evidence; a ✗ is a
> prompt to locate the discrepancy, not an automatic verdict. Grid energies carry
> the stated $O(h^2)$ offset from the exact Hartree–Fock limits (about $10^{-3}$
> relative for beryllium's compact $1s$); the gates say which comparison is meant.
>
> **Scope.** Restricted, closed-shell, $s$-only: the full Roothaan machinery
> (basis sets, open shells, higher angular momenta) belongs to quantum-chemistry
> texts — Szabo & Ostlund {cite}`szabo1996`, Ch. 3, is the canonical reference and
> carries every derivation sketched here to completion. Koopmans' original paper is
> {cite}`koopmans1934`; the Hartree–Fock limits quoted are the standard numerical
> values (Szabo & Ostlund, Table 3.6 and references therein).

## Theory in brief

### The energy of a Slater determinant

A determinant $\Phi = \mathrm{det}[\phi_1\cdots\phi_N]/\sqrt{N!}$ of orthonormal
spin-orbitals is the antisymmetrized product state of
[§6.20](../06-quantum-mechanics/identical-particles.ipynb). Taking the expectation
value of the Hamiltonian in it is bookkeeping with a beautiful outcome (the
Slater–Condon rules; Szabo & Ostlund {cite}`szabo1996`, §2.3, work every case): the
one-body terms sum over occupied orbitals, and the two-body repulsion contributes
*two* kinds of integral,

```{math}
:label: eq-hf-energy
E[\{\phi\}] = \sum_i \langle \phi_i | \hat h | \phi_i \rangle
+ \tfrac12 \sum_{ij} \big( J_{ij} - K_{ij}\,\delta_{\sigma_i\sigma_j} \big),
\qquad
\begin{aligned}
J_{ij} &= \iint \frac{|\phi_i(\mathbf r)|^2 |\phi_j(\mathbf r')|^2}{|\mathbf r - \mathbf r'|}, \\
K_{ij} &= \iint \frac{\phi_i^*(\mathbf r)\phi_j(\mathbf r)\,\phi_j^*(\mathbf r')\phi_i(\mathbf r')}{|\mathbf r - \mathbf r'|}.
\end{aligned}
```

$J_{ij}$ is classical: the Coulomb energy of two charge clouds, the term Hartree
wrote down by electrostatic intuition in 1928. $K_{ij}$ has no classical picture:
it swaps the orbitals between the two integration points, exists only between
*like-spin* pairs (the $\delta_{\sigma_i\sigma_j}$), and enters with a minus sign —
antisymmetry keeps like-spin electrons apart, lowering their repulsion. Note
$K_{ii} = J_{ii}$: the exchange term exactly cancels the unphysical interaction of
an electron with itself, a bookkeeping grace [§8.7](kohn-sham.ipynb) will find
painfully absent in approximate functionals.

### The Hartree–Fock equations, Koopmans, Brillouin

Minimizing Eq. {eq}`eq-hf-energy` over orbitals, with Lagrange multipliers
enforcing orthonormality (the same constrained-variation pattern as
[§6.22](../06-quantum-mechanics/variational-method.ipynb)), yields the canonical
Hartree–Fock equations

```{math}
:label: eq-hf-fock
\hat F \phi_i = \varepsilon_i \phi_i,
\qquad
\hat F = \hat h + \sum_j \big( \hat J_j - \hat K_j\,\delta_{\sigma\sigma_j} \big),
```

where $\hat J_j$ multiplies by the potential of orbital $j$'s charge and $\hat K_j$
is the *nonlocal* exchange operator,
$(\hat K_j \phi)(\mathbf r) = \phi_j(\mathbf r)\!\int \phi_j^*(\mathbf r')\phi(\mathbf r')/|\mathbf r - \mathbf r'|\,d\mathbf r'$:
its action at $\mathbf r$ requires $\phi$ *everywhere*. Because $\hat F$ depends on
its own eigenfunctions, the equations demand self-consistency, the fixed-point loop
of [§8.2](exact-laboratory.ipynb) Exercise 6.

Two theorems ride along free. **Koopmans**: removing an electron from orbital $i$
*without letting the others relax* costs exactly $-\varepsilon_i$ — so occupied
orbital energies approximate ionization energies, with two neglected effects
(orbital relaxation, which lowers the true cost, and correlation, which raises it)
partially cancelling {cite}`koopmans1934`. **Brillouin**: the converged determinant
has vanishing Hamiltonian matrix elements with all single excitations — the
variational optimum has used up everything one-orbital changes can offer, so the
missing physics (correlation) begins at *double* excitations. Both are proved in
three lines from Eq. {eq}`eq-hf-energy`; Szabo & Ostlund {cite}`szabo1996`, §3.3,
spell them out.

### The radial reduction, and this notebook's atoms

For closed-shell atoms with only $s$ orbitals, every $\phi_i$ is spherically
symmetric, $\phi = u(r)/(r\sqrt{4\pi})$, and the Coulomb kernel collapses to its
monopole: $1/|\mathbf r - \mathbf r'| \to 1/r_>$ after angular integration (the
quantum shell theorem — a sphere of charge acts from its center). The working
equations on the radial grid are then

```{math}
:label: eq-hf-radial
J_{ij} = \iint u_i^2(r)\, \frac{1}{r_>}\, u_j^2(r')\,dr\,dr',
\qquad
(\hat K_j u)(r) = u_j(r) \int u_j(r')\,\frac{1}{r_>}\,u(r')\,dr' ,
```

and the whole Fock operator is an $N \times N$ matrix: tridiagonal kinetic part,
diagonal external and Hartree potentials, and the exchange as the dense rank-mixing
matrix $X_{ab} = \sum_j u_j(r_a)\,(1/r_>)_{ab}\,u_j(r_b)\,h$. Helium ($1s^2$,
opposite spins) has the special grace that its only exchange integral is $K_{11} =
J_{11}$, making the effective potential local; beryllium ($1s^2 2s^2$) has genuine
$1s$–$2s$ exchange and requires the full nonlocal matrix — which is precisely why it
is in this notebook.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import eigh, eigh_tridiagonal

from ecp import validate

INK, AMBER, SOFT = "#16213e", "#c0851a", "#46506b"
HARTREE_EV = 27.211386  # 1 Ha in eV, the section 8.1 conversion

# The radial grid, fixed for the notebook
N_R, R_MAX = 1400, 25.0
H_R = R_MAX / N_R
r = H_R * np.arange(1, N_R + 1)

# monopole Coulomb kernel 1/r_> as a full matrix (the shell theorem on the grid)
K_MONO = 1.0 / np.maximum.outer(r, r)

# kinetic + external pieces reused throughout
T_KIN = (
    np.diag(np.full(N_R, 1.0 / H_R**2))
    + np.diag(np.full(N_R - 1, -0.5 / H_R**2), 1)
    + np.diag(np.full(N_R - 1, -0.5 / H_R**2), -1)
)


def hydrogenic_1s(Z):
    """The analytic hydrogenic 1s reduced radial function u(r) = 2 Z^(3/2) r e^(-Zr).

    The exact ground orbital of a bare charge-Z nucleus, normalized on the
    notebook's grid; used as the frozen trial orbital and as every SCF start.

    Parameters
    ----------
    Z : float
        Nuclear charge.

    Returns
    -------
    numpy.ndarray
        u(r) on the grid, normalized so that the trapezoid of u^2 equals 1.
    """
    u = 2.0 * Z**1.5 * r * np.exp(-Z * r)
    return u / np.sqrt(np.trapezoid(u**2, r))


def coulomb_J(u_i, u_j):
    """The radial Coulomb integral J_ij of Eq. eq-hf-radial.

    The classical repulsion energy of the charge shells u_i^2 and u_j^2
    interacting through the monopole kernel 1/r_>.

    Parameters
    ----------
    u_i, u_j : numpy.ndarray
        Reduced radial orbitals on the grid, each normalized to 1.

    Returns
    -------
    float
        J_ij in Hartree.
    """
    return H_R * H_R * (u_i**2) @ K_MONO @ (u_j**2)


def exchange_K(u_i, u_j):
    """The radial exchange integral K_ij of Eq. eq-hf-radial.

    The orbital-swapped integral born from antisymmetry; K_ii = J_ii (the
    self-interaction cancellation), and K_12 between different shells is the
    genuinely nonlocal content of Hartree-Fock.

    Parameters
    ----------
    u_i, u_j : numpy.ndarray
        Reduced radial orbitals on the grid, each normalized to 1.

    Returns
    -------
    float
        K_ij in Hartree.
    """
    return H_R * H_R * (u_i * u_j) @ K_MONO @ (u_i * u_j)

## Exercise 1 — The determinant's energy, certified against closed forms

Before any self-consistency, the machinery of Eq. {eq}`eq-hf-energy` must prove
itself on integrals whose values are known exactly. For hydrogenic $1s$ orbitals
the two building blocks have famous closed forms: the one-body energy in a bare
charge-$Z$ potential is $\langle \hat h \rangle = Z^2/2 - Z^2 = -Z^2/2$, and the
Coulomb self-repulsion is $J_{1s,1s} = \tfrac58 Z$ — the integral behind the
$27/8$ of the helium curve in [§8.1](many-electron-problem.ipynb) (Szabo &
Ostlund {cite}`szabo1996`, §1.3, derive it). Their combination for helium with
*frozen* $Z = 2$ orbitals must reproduce $E(2) = 2^2 - \tfrac{27}{8}\cdot 2 =
-2.75$ Ha of Eq. {eq}`eq-me-hylleraas` exactly.

**Part a)** Build the analytic hydrogenic $u_{1s}$ for $Z = 2$ with the Setup
helper, and evaluate $J_{11}$ with `coulomb_J` (the double `numpy` contraction of
the monopole kernel). Compare against $\tfrac58 Z = 1.25$ Ha.

**Part b)** Evaluate $\langle u |\hat h| u\rangle$ as the quadratic form of the
kinetic matrix plus the $-Z/r$ diagonal (`numpy` matrix–vector product and
`numpy.trapezoid`-consistent grid weights), compare against $-Z^2/2 = -2$ Ha, and
assemble the frozen-orbital helium energy $E = 2\langle h\rangle + J_{11}$ against
the closed-form $-2.75$ Ha. With the integrals certified, everything after runs on
trusted ground.

In [ ]:
# (solution hidden on the public site)


### Validation 1 — the integrals earn trust

$J_{11}$ must match $5Z/8$ and the frozen-orbital energy the closed-form $-2.75$
Ha, both to the grid's $O(h^2)$ accuracy.

In [ ]:
validate.close(J_frozen, 1.25, "hydrogenic Coulomb integral J = 5Z/8", rtol=5e-4)
validate.close(one_frozen, -2.0, "hydrogenic one-body energy -Z^2/2", rtol=5e-4)
validate.close(E_frozen, -2.75, "frozen-orbital helium energy", rtol=1e-3)

## Exercise 2 — Helium, self-consistently

Now the orbitals are released. For helium's $1s^2$ the two spins are opposite, so
the only exchange integral is $K_{11} = J_{11}$ and Eq. {eq}`eq-hf-fock` collapses
to a *local* effective potential: each electron feels the nucleus plus the mean
field of exactly one companion,
$\hat F = \hat h + \hat J_1$ (the $2J - K = J$ arithmetic of the pair, exactly as
in the laboratory of [§8.2](exact-laboratory.ipynb)). The self-consistent loop is:
build $v_H(r) = h\,(1/r_> \mathbin{@}\, u^2)$ from the current orbital (the `@`
product of the monopole kernel with the squared orbital), rediagonalize
$\hat h + v_H$, repeat.

**Part a)** Iterate from the hydrogenic $Z = 2$ start using `scipy.linalg.eigh` on
the dense Fock matrix until the total energy
$E = 2\langle u|\hat h|u\rangle + J_{11}$ changes by under $10^{-9}$ Ha, tracking
the energy change per iteration. Report $E_{\mathrm{HF}}$, $\varepsilon_{1s}$, and
the iteration count; the literature Hartree–Fock limit is $-2.8617$ Ha.

**Part b)** Verify the virial theorem $\langle V\rangle/\langle T\rangle = -2$
(compute $\langle T\rangle$ as the kinetic quadratic form, $\langle V\rangle =
E - \langle T\rangle$): the classic health check of a converged SCF, sensitive to
almost any implementation error. Plot the convergence trace on a log axis.

In [ ]:
# (solution hidden on the public site)


### Validation 2 — helium's certificate

The grid Hartree–Fock energy is $-2.8606$ Ha (the exact HF limit $-2.8617$ minus
the grid's $O(h^2)$ offset; both comparisons gated), the orbital energy $-0.9176$
Ha against the literature $-0.9180$, and the virial ratio $-2$ to a part in a
thousand.

In [ ]:
validate.close(E_he, -2.86062, "helium HF energy on this grid", rtol=1e-4)
validate.close(E_he, -2.8617, "helium HF energy vs the exact HF limit", rtol=1e-3)
validate.close(
    eps_1s_he, -0.9180, "helium orbital energy vs the HF-limit value", rtol=1e-3
)
validate.close(virial_he, -2.0, "the virial theorem at self-consistency", rtol=2e-3)

## Exercise 3 — Screening, made visible

[§8.1](many-electron-problem.ipynb) *told* the screening story through one number,
$Z_{\mathrm{eff}} = 27/16$; the converged orbital now *shows* it. Three $1s$
orbitals share one plot: the bare hydrogenic $Z = 2$ (what each electron would be
without its companion), the screened hydrogenic $Z = 27/16$ (the best
single-parameter compromise), and the self-consistent Hartree–Fock orbital (the
best *any* radial shape can do). Each has a mean radius; the exact hydrogenic
expectation $\langle r\rangle = 3/(2Z)$ from
[§6.17](../06-quantum-mechanics/hydrogen-atom.ipynb) anchors the first two.

**Part a)** Compute $\langle r\rangle = \int r\,u^2\,dr$ (`numpy.trapezoid`) for
the three orbitals: hydrogenic $Z = 2$ (expect $0.750$), hydrogenic $Z = 27/16$
(expect $0.889$), and the converged HF orbital of Exercise 2. The HF cloud should
be the *largest* of the three: self-consistent screening is stronger at large $r$
than any single effective charge, because the far reaches of the orbital see the
companion's charge almost completely.

**Part b)** Plot the three $u(r)$ together. The HF orbital should hug the
screened curve near the nucleus and exceed both hydrogenic tails beyond
$r \approx 2$ Bohr: screening is *shape*, not just scale.

In [ ]:
# (solution hidden on the public site)


### Validation 3 — the cloud swells in the right order

The hydrogenic means must match $3/(2Z)$, and the ordering
$\langle r\rangle_{Z=2} < \langle r\rangle_{27/16} < \langle r\rangle_{\mathrm{HF}}$
must hold, with the HF value at its measured $0.928$ Bohr.

In [ ]:
validate.close(r_means["bare Z=2"], 0.75, "hydrogenic <r> at Z = 2", rtol=1e-3)
validate.close(
    r_means["screened 27/16"],
    3.0 / (2 * 27.0 / 16.0),
    "hydrogenic <r> at 27/16",
    rtol=1e-3,
)
validate.close(r_means["Hartree-Fock"], 0.9276, "the HF mean radius", rtol=1e-3)
validate.check(
    r_means["bare Z=2"] < r_means["screened 27/16"] < r_means["Hartree-Fock"],
    "screening ordering of the three clouds",
    "bare < screened < self-consistent",
)

## Exercise 4 — Beryllium: nonlocal exchange and the birth of shells

Helium never needed the full Fock operator; beryllium does. With $1s^2 2s^2$ there
are two occupied spatial orbitals, and the like-spin $1s$–$2s$ pairs generate a
genuine exchange coupling: the operator $\hat K$ of Eq. {eq}`eq-hf-fock`, on the
grid the dense matrix $X_{ab} = \sum_j u_j(r_a)\,(1/r_>)_{ab}\,u_j(r_b)\,h$
(assembled as the `numpy` outer-product sum $U U^{\mathsf T}$ multiplied
elementwise by the kernel). Diagonalizing $T + \mathrm{diag}(v_{\mathrm{ext}} +
v_H) - X$ with `scipy.linalg.eigh` and re-occupying the two lowest orbitals closes
the loop. Two things are being tested at once: the machinery (against the
literature limit $E_{\mathrm{HF}} = -14.573$ Ha) and the *concept* of shells,
which nothing put in by hand — the $2s$ orbital, its single node, and the factor
of six between the shells' radii all emerge from self-consistency plus Pauli.

**Part a)** Run the beryllium SCF: Hartree potential from the *total* density
$2\sum_j u_j^2$ (all four electrons), exchange matrix from both occupied orbitals,
`scipy.linalg.eigh` per iteration, converging the total energy
$E = \sum_i 2\langle i|\hat h|i\rangle + \sum_{ij}(2J_{ij} - K_{ij})$ (the Setup
helpers `coulomb_J` and `exchange_K`) to $10^{-9}$ Ha. Report $E$,
$\varepsilon_{1s}$, $\varepsilon_{2s}$ against the literature values
$-14.573$, $-4.7327$, $-0.3093$ Ha.

**Part b)** Exhibit the shells: plot $u_{1s}$ and $u_{2s}$ and the radial density
$4\pi r^2 n \to 2(u_{1s}^2 + u_{2s}^2)$; count the $2s$ node (`numpy.sign`
changes away from the endpoints); report $\langle r\rangle_{1s} = 0.415$ and
$\langle r\rangle_{2s} = 2.65$ Bohr. The two-humped radial density *is* the shell
structure of the periodic table, computed from first principles.

In [ ]:
# (solution hidden on the public site)


### Validation 4 — beryllium's certificate

The total energy must land on the literature HF limit to the grid's accuracy
(the compact $1s$ costs about $10^{-3}$ relative at $h = 0.018$; both the grid
value and the limit comparison are gated), the orbital energies on their
literature values, the $2s$ must carry exactly one node, and the shell radii must
sit at $0.415$ and $2.65$ Bohr.

In [ ]:
validate.close(E_be, -14.5538, "beryllium HF energy on this grid", rtol=1e-4)
validate.close(E_be, -14.5730, "beryllium HF energy vs the exact HF limit", rtol=2e-3)
validate.close(eps_1s_be, -4.7327, "eps_1s vs the HF-limit value", rtol=2e-3)
validate.close(eps_2s_be, -0.3093, "eps_2s vs the HF-limit value", rtol=1e-3)
validate.check(
    nodes_2s == 1, "the 2s orbital carries exactly one node", f"{nodes_2s} nodes"
)
validate.close(r_1s, 0.4154, "the K-shell mean radius", rtol=1e-3)
validate.close(r_2s, 2.6505, "the L-shell mean radius", rtol=1e-3)

## Exercise 5 — Koopmans meets the experiment

Koopmans' theorem prices an ionization at $-\varepsilon_i$, frozen orbitals and
all. The National Institute of Standards and Technology tables give the measured
first ionization energies: helium $24.587$ eV, beryllium $9.323$ eV. The
comparison is a two-line computation and a lesson in error anatomy: for helium the
frozen-orbital error (overestimates $I$: the ion would relax) and the correlation
error (underestimates $I$: the neutral had more correlation to lose) nearly cancel,
landing within $1.6\%$; for beryllium's soft $2s$ shell the relaxation is
relatively larger and Koopmans lands $10\%$ low — the theorem is a first estimate,
not a guarantee.

**Part a)** Convert $-\varepsilon_{1s}$(He) and $-\varepsilon_{2s}$(Be) to eV with
the [§8.1](many-electron-problem.ipynb) conversion constant and compare against the
measured values (`numpy` arithmetic; report signed percent errors).

**Part b)** Chart the comparison (a `matplotlib` grouped bar chart, Koopmans vs
measured for the two atoms) with the signed errors annotated.

In [ ]:
# (solution hidden on the public site)


### Validation 5 — the two error anatomies

Helium's Koopmans estimate must land within $2\%$ of the measurement (the
cancellation), beryllium's must be $8$–$12\%$ *low* (relaxation dominating):
both the accuracy and the failure are the lesson.

In [ ]:
validate.check(
    abs(errors["He"]) < 0.02,
    "Koopmans within 2% for helium",
    f"{100 * errors['He']:+.2f}%",
)
validate.check(
    -0.12 < errors["Be"] < -0.08,
    "Koopmans 8-12% low for beryllium",
    f"{100 * errors['Be']:+.2f}%",
)

## Exercise 6 — The correlation ledger of real atoms

[§8.2](exact-laboratory.ipynb) measured the laboratory's correlation energy;
now the same ledger opens for real atoms, with our own Hartree–Fock numbers on one
side and the exact non-relativistic energies (helium $-2.90372$, beryllium
$-14.6674$ Ha; Parr & Yang {cite}`parryang1989`, App., and references therein) on
the other. The percentages stay at the percent level; the absolute numbers bite: beryllium's
correlation energy is $2.6$ eV, a chemical bond's worth of physics invisible to
the best determinant.

**Part a)** Assemble the ledger: for He and Be, the computed
$E_{\mathrm{HF}}$ (grid values, with the HF-limit values alongside), the exact
energies, $E_c = E_{\mathrm{exact}} - E_{\mathrm{HF\ limit}}$, and $E_c$ both as a
percentage of $E$ and in eV.

**Part b)** Draw the energy ladder for each atom (frozen $Z$-orbital estimate
where available, HF, exact — a `matplotlib` horizontal-line ladder): the visual
summary of what self-consistency buys and what it cannot.

In [ ]:
# (solution hidden on the public site)


### Validation 6 — the ledger's totals

The correlation energies must come out $-0.0420$ Ha (helium) and $-0.0944$ Ha
(beryllium), and both must sit at the percent level of the total energy while
exceeding one eV: the sliver that decides chemistry.

In [ ]:
validate.close(E_c_atoms["He"], -0.0420, "helium correlation energy", rtol=2e-2)
validate.close(E_c_atoms["Be"], -0.0944, "beryllium correlation energy", rtol=2e-2)
validate.check(
    all(abs(E_c_atoms[a] / E_EXACT[a]) < 0.02 for a in ("He", "Be")),
    "correlation stays at the percent level of the total energy",
    "1.45% and 0.64%",
)
validate.check(
    all(abs(E_c_atoms[a]) * HARTREE_EV > 1.0 for a in ("He", "Be")),
    "and yet exceeds an electronvolt",
    f"{abs(E_c_atoms['He']) * HARTREE_EV:.2f} and {abs(E_c_atoms['Be']) * HARTREE_EV:.2f} eV",
)

:::{admonition} With your assistant
:class: tip
The beryllium SCF loop generalizes to any closed-shell $s$-only atom by changing
$Z$ and the orbital count. Have your assistant extend it to magnesium-like
$1s^2 2s^2 3s^2$ at $Z = 12$ (a fiction — real magnesium fills $2p$ first — but a
well-posed one), then run the check that is yours alone: the virial ratio
$\langle V\rangle/\langle T\rangle$ must return $-2$ within $10^{-2}$, and the
three orbitals must carry $0, 1, 2$ nodes respectively. The check is yours.
:::

## Notebook summary

Hartree–Fock is now a derived, working method rather than a name. The determinant's
energy resolved into one-body, Hartree, and exchange pieces, and the radial
integrals certified themselves against closed forms ($J_{1s,1s} = 5Z/8$ to
$3\times10^{-4}$; frozen-orbital helium at $-2.75$ Ha on the nose). Self-consistent
helium converged in seven iterations to $E_{\mathrm{HF}} = -2.8606$ Ha on the grid
(limit $-2.8617$), with the virial ratio $-2.0007$ certifying the implementation
and the converged orbital exhibiting screening as *shape*
($\langle r\rangle = 0.928$ Bohr, beyond both hydrogenic references). Beryllium
brought the genuinely nonlocal exchange matrix and returned $-14.554$ Ha
(limit $-14.573$), orbital energies $-4.725/-0.3091$ Ha, and shells nobody put in:
a one-node $2s$ at $\langle r\rangle = 2.65$ Bohr over a $1s$ core at $0.42$.
Koopmans' theorem met experiment with both its faces (helium $+1.6\%$ by error
cancellation; beryllium $-9.8\%$, relaxation winning), and the correlation ledger
closed at $-0.042$ Ha (He) and $-0.094$ Ha (Be): a percent-level sliver of the energy,
more than an eV of chemistry.

## Outlook

- Exchange was cheap here because atoms are small. In extended systems the
  nonlocal operator becomes the bottleneck, and in the homogeneous electron gas it
  can be evaluated *exactly*, exposing both the origin of the local-density
  approximation and a genuine Hartree–Fock pathology at the Fermi surface:
  [§8.4](hartree-fock-gas.ipynb).
- The correlation energies tabulated here are targets. Configuration interaction,
  many-body perturbation theory, and coupled cluster attack them head-on (Szabo &
  Ostlund {cite}`szabo1996`, Ch. 4–6); this volume's road goes instead through the
  density: [§8.5](thomas-fermi.ipynb) through [§8.8](exact-conditions-band-gap.ipynb).
- Brillouin's theorem (singles do nothing) is why the simplest correlation theory
  starts at second order in doubles — the Møller–Plesset ladder, met again as the
  perturbative rung of [§8.13](hubbard-model.ipynb)'s comparisons.

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()